In [2]:
import os

import cv2
import easyocr
import matplotlib.pyplot as plt
import numpy as np
import openpyxl

In [57]:
def recognize_number(img, reader):
    y, x, _ = img.shape
    if x > y:
        img = img = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
    
    for _ in range(4):
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        padded = cv2.copyMakeBorder(gray, 30, 30, 30, 30, cv2.BORDER_CONSTANT, value=255)
        
        recognized = reader.readtext(padded, detail=0, paragraph=False, text_threshold=0.9, allowlist="0123456789")
        if recognized:
            recognized = list(filter(lambda x: len(x) == 6, [s for s in recognized]))
            if recognized:
                break
        
        img = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)    

    return recognized[0] if recognized else 'Не распознано'


In [58]:

reader = easyocr.Reader(["en"])
images_dir = "./images"

recognized = 0
amount = 0
unrecognized = []

wb = openpyxl.Workbook()
ws = wb.active
ws.append(["Файл", "Номер"])

for filename in sorted(os.listdir(images_dir)):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        photo_id = os.path.splitext(filename)[0]
        
        with open(os.path.join(images_dir, filename), 'rb') as f:
            img = np.asarray(bytearray(f.read()), dtype=np.uint8)
            img = cv2.imdecode(img, cv2.IMREAD_COLOR)

        if img is not None:
            amount += 1
            
            number: str = recognize_number(img, reader)
            print(f"{photo_id}: {number}")
            
            if number.isnumeric():
                recognized += 1
                ws.append([filename, number])
            else:
                unrecognized.append(filename)
                ws.append([filename, ""])

wb.save("results.xlsx")
print(f'Всего: {amount}, из которых {recognized} распознаны')



1ф. 066894+: 066894
1ф. 066905+: 066905
1ф. 066911+: 006691
1ф. 066916+: 916990
1ф. 066917+: 202755
1ф. 066959+: 684457
1ф. 067126+: 067126
1ф. 067219+: 612290
1ф. 067220+: 144529
1ф. 067279+: 622290
1ф. 067287+: 067287
1ф. 067449+: 484340
1ф. 067484+: 562516
1ф. 067509+: 207267
1ф. 067574+: 067574
1ф. 067652+: 067652
1ф. 067653+: 849290
1ф. 067756+: 100055
1ф. 067758+: 216133
1ф. 067759+: 067759
1ф. 067760+: Не распознано
1ф. 067783+: 053251
1ф. 067794+: 819183
1ф. 067795+: 816133
1ф. 067986+: 986290
1ф. 068028+: 820890
1ф. 068029+: 620890
1ф. 068034+: 496100
1ф. 068218+: 068218
1ф. 068225+: 422890
371412: 304704
371414: 301729
374366: 305061
390245: 322894
390255: 008990
390818: 023491
390819: 023493
390823: 023496
390824: 023522
392297: 334421
392303: 442974
392306: 443036
392310: 443282
392312: 443438
392318: 443451
392320: 443452
392324: 443455
392327: 443462
392331: 443472
392333: 443493
392498: 303349
392502: 310582
392507: 810831
392510: 310876
392514: 364318
392516: 364603
392

In [68]:
unrecognized

['1ф. 067760+.jpg', '3ф. 016063+.jpg', '3ф. 016146+.jpg']